# Adult Census Income: EDA

В этом notebook выполняется exploratory data analysis для очищенного Adult Census Income dataset.

Цель этапа:

- понять распределение целевой переменной;
- изучить числовые и категориальные признаки;
- найти признаки, потенциально связанные с доходом;
- подготовить идеи для feature engineering и preprocessing.

Обучение моделей и feature engineering здесь пока не выполняются.

## Загрузка очищенных данных

Базовая загрузка и очистка вынесены в `adult_income_utils.py`, чтобы не дублировать один и тот же код в каждом notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))
sys.path.append(str(Path.cwd() / "src"))

from adult_income_utils import load_clean_adult_data

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

In [ ]:
df_clean = load_clean_adult_data()
df_eda = df_clean.fillna("Unknown")

print("Rows after cleaning:", len(df_clean))
df_eda.head()

`df_clean` сохраняет пропуски как `NaN`. Для EDA-графиков используем `df_eda`, где пропуски заменены на `Unknown`, чтобы неизвестные категории были видны на графиках.

В финальном ML-пайплайне заполнение пропусков всё равно должно выполняться внутри `sklearn Pipeline`.

## Общий обзор данных

In [ ]:
print("Shape:", df_eda.shape)

df_eda.info()

In [ ]:
numeric_columns = df_eda.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df_eda.select_dtypes(include="object").columns.tolist()

print("Numeric columns:", numeric_columns)
print("Categorical columns:", categorical_columns)

In [ ]:
df_eda[numeric_columns].describe().T

In [ ]:
df_eda[categorical_columns].nunique().sort_values(ascending=False)

Признак `fnlwgt` означает final weight: это технический вес записи в выборке Census, а не характеристика человека вроде возраста или образования. Его можно оставить как числовой признак, но интерпретировать нужно осторожно.

## Распределение целевой переменной

In [ ]:
target_distribution = pd.DataFrame({
    "count": df_eda["income"].value_counts(),
    "percent": df_eda["income"].value_counts(normalize=True) * 100,
})

target_distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df_eda, x="income", order=["<=50K", ">50K"])
plt.title("Income class distribution")
plt.xlabel("Income")
plt.ylabel("Count")
plt.show()

Классы несбалансированы: объектов с доходом `<=50K` заметно больше, чем объектов с доходом `>50K`. Поэтому в моделировании нельзя полагаться только на `accuracy`.

## Распределения числовых признаков

In [ ]:
df_eda[numeric_columns].hist(figsize=(14, 8), bins=30)
plt.suptitle("Numeric feature distributions", y=1.02)
plt.tight_layout()
plt.show()

`capital_gain` и `capital_loss` имеют сильный перекос: у большинства объектов значения равны нулю. Это может быть полезно для feature engineering: например, можно добавить признаки факта наличия capital gain/loss.

## Числовые признаки и доход

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.boxplot(data=df_eda, x="income", y="age", order=["<=50K", ">50K"], ax=axes[0])
axes[0].set_title("Age by income")

sns.boxplot(data=df_eda, x="income", y="education_num", order=["<=50K", ">50K"], ax=axes[1])
axes[1].set_title("Education years by income")

sns.boxplot(data=df_eda, x="income", y="hours_per_week", order=["<=50K", ">50K"], ax=axes[2])
axes[2].set_title("Hours per week by income")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(data=df_eda, x="capital_gain", hue="income", bins=40, ax=axes[0])
axes[0].set_title("Capital gain by income")

sns.histplot(data=df_eda, x="capital_loss", hue="income", bins=40, ax=axes[1])
axes[1].set_title("Capital loss by income")

plt.tight_layout()
plt.show()

## Категориальные признаки и доход

Для категориальных признаков удобно смотреть долю класса `>50K` внутри каждой категории.

In [ ]:
def plot_high_income_rate(column, top_n=None, figsize=(9, 5)):
    data = df_eda.copy()

    if top_n is not None:
        top_categories = data[column].value_counts().head(top_n).index
        data = data[data[column].isin(top_categories)]

    income_rate = pd.crosstab(data[column], data["income"], normalize="index")
    income_rate = income_rate[">50K"].sort_values(ascending=True)

    plt.figure(figsize=figsize)
    sns.barplot(x=income_rate.values, y=income_rate.index)
    plt.title(f">50K share by {column}")
    plt.xlabel("Share of >50K")
    plt.ylabel(column)
    plt.xlim(0, 1)
    plt.show()

    return income_rate.to_frame(name=">50K_share")

In [ ]:
plot_high_income_rate("education", figsize=(9, 6))

In [ ]:
plot_high_income_rate("marital_status", figsize=(9, 5))

In [ ]:
plot_high_income_rate("occupation", top_n=12, figsize=(9, 6))

In [ ]:
plot_high_income_rate("workclass", figsize=(9, 5))

Видно, что доля класса `>50K` различается между категориями образования, семейного статуса, профессии и типа занятости. Поэтому категориальные признаки важно кодировать, а не удалять.

## Чувствительные признаки

`sex` и `race` могут быть связаны с целевой переменной, но эти признаки нужно интерпретировать осторожно. Adult Census Income отражает социально-экономические данные 1994 года и может содержать исторические смещения.

In [ ]:
plot_high_income_rate("sex", figsize=(7, 3))

In [ ]:
plot_high_income_rate("race", figsize=(8, 4))

## Корреляции числовых признаков

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df_eda[numeric_columns].corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Numeric feature correlations")
plt.show()

Корреляции между числовыми признаками в целом не выглядят критично высокими. При этом `education` и `education_num` описывают почти одну и ту же информацию: один признак категориальный, другой числовой.

## Предварительные выводы

- В данных есть дисбаланс классов: `<=50K` встречается заметно чаще, чем `>50K`.
- Для оценки моделей нужны метрики помимо accuracy: `balanced_accuracy`, `precision`, `recall`, `f1`, `roc_auc`.
- Числовые признаки имеют разные масштабы, поэтому для Logistic Regression и KNN понадобится scaling.
- Категориальные признаки важны, поэтому их нужно кодировать через OneHotEncoder.
- `capital_gain` и `capital_loss` сильно скошены и часто равны нулю; это хорошая основа для простого feature engineering.
- `age`, `education`, `occupation`, `marital_status`, `hours_per_week` и capital-признаки выглядят потенциально полезными для предсказания дохода.
- `fnlwgt` является техническим весом записи, поэтому его интерпретация требует осторожности.